In [1]:
import pandas as pd

In [2]:
# Goal is to use Maven cases and their data to create UAT test cases
# The maven data is the file below

In [3]:
# Read maven case data
df_maven = pd.read_csv('sql_scripts\query_results\cases_with_values_20260626.csv',na_filter=False)
len(df_maven)

<>:2: SyntaxWarning: invalid escape sequence '\q'
<>:2: SyntaxWarning: invalid escape sequence '\q'
C:\Users\s.chintabathina\AppData\Local\Temp\ipykernel_46628\683054303.py:2: SyntaxWarning: invalid escape sequence '\q'
  df_maven = pd.read_csv('sql_scripts\query_results\cases_with_values_20260626.csv',na_filter=False)


110194

In [4]:
df_maven.head()

,case_id,case_modified_date,first_name,last_name,birth_date,model_name,QUESTIONSET_ID,QUESTION_ID,value
0,100362493,2026-04-16,Champ,Wright,1959-12-21,DiseaseSurveillanceModel_Hep,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 MSH MSH.1=| MSH.2=^~\& MSH.3 HD.1=LAB ...
1,100362493,2026-04-16,Champ,Wright,1959-12-21,DiseaseSurveillanceModel_Hep,ADMINISTRATIVE,ELR_REVIEWED_NO,NO
2,100362493,2026-04-16,Champ,Wright,1959-12-21,DiseaseSurveillanceModel_Hep,ADMINISTRATIVE,ADDITIONAL_SOURCE_CASE_REPORT_ELR,ELR
3,100362493,2026-04-16,Champ,Wright,1959-12-21,DiseaseSurveillanceModel_Hep,ADMINISTRATIVE,REPORTING_SOURCE_DATE,06/20/2014
4,100362493,2026-04-16,Champ,Wright,1959-12-21,DiseaseSurveillanceModel_Hep,ADMINISTRATIVE,REPORTING_SOURCE_TYPE,PHC247


In [5]:
def clean_up(name):
    new_name =''
    for ch in name:
        # If space, digits, uppercase letters, lowercase letters, or underscore, add it to new name otherwise ignore
        if (ord(ch)==32 or 48<=ord(ch)<=57 or 65<=ord(ch)<=90 or 97<=ord(ch)<=122 or ord(ch)==95 ):
            new_name+=ch
    return new_name.strip()

In [6]:
# Clean up questions or data elements
df_maven.loc[:,'QUESTION_ID'] =[ clean_up(q) for q in df_maven.loc[:,'QUESTION_ID']]

In [7]:
df_maven.loc[:,'model_name'].value_counts(dropna=False)

model_name
DiseaseSurveillanceModel_General    69278
DiseaseSurveillanceModel_STD        23669
DiseaseSurveillanceModel_Hep        14908
Childhood_Lead_Child_Model           2339
Name: count, dtype: int64

In [8]:
# General clean up
df_maven.loc[:,'model_name'] = [m.strip() for m in df_maven.loc[:,'model_name']]

In [9]:
# Rename model names
df_maven.loc[:,'model_name'] = [ m.split('_')[1] if 'DiseaseSurveillanceModel_' in m else m for m in df_maven.loc[:,'model_name']]

In [10]:
df_maven.loc[:,'model_name'].value_counts(dropna=False)

model_name
General                       69278
STD                           23669
Hep                           14908
Childhood_Lead_Child_Model     2339
Name: count, dtype: int64

In [11]:
# Using same model names as uat document
df_maven.loc[:,'model_name'] = ['Hepatitis' if m=="Hep" else 'Childhood Lead' if m=='Childhood_Lead_Child_Model' else m for m in df_maven.loc[:,'model_name']]

In [12]:
df_maven.loc[:,'model_name'].value_counts(dropna=False)

model_name
General           69278
STD               23669
Hepatitis         14908
Childhood Lead     2339
Name: count, dtype: int64

In [13]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:]==answer):
        return question[:start].strip('_')
    else:
        return question

In [14]:
# Truncate question if answer is contained.
# If last part of question equals value then truncate the question
df_maven.loc[:,'QUESTION_ID']  = [ join_all_except_last(q,a) for q,a in zip(df_maven.loc[:,'QUESTION_ID'],df_maven.loc[:,'value'])]

In [15]:
df_maven

,case_id,case_modified_date,first_name,last_name,birth_date,model_name,QUESTIONSET_ID,QUESTION_ID,value
0,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 MSH MSH.1=| MSH.2=^~\& MSH.3 HD.1=LAB ...
1,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,ELR_REVIEWED,NO
2,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,ADDITIONAL_SOURCE_CASE_REPORT,ELR
3,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_DATE,06/20/2014
4,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_TYPE,PHC247
...,...,...,...,...,...,...,...,...,...
110189,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE_DAYS,7787
110190,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE,21
110191,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE_UNITS,YEARS
110192,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,ADMINISTRATIVE,MMWR_WEEK,21


In [16]:
# duplicates
df_maven[df_maven.duplicated(['case_id','QUESTION_ID'])]

,case_id,case_modified_date,first_name,last_name,birth_date,model_name,QUESTIONSET_ID,QUESTION_ID,value
37,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 MSH MSH.1=| MSH.2=^~\& MSH.3 HD.1=LAB ...
38,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_DATE,06/20/2014
39,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_TYPE,PHC247
45,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 MSH MSH.1=| MSH.2=^~\& MSH.3 HD.1=LAB ...
46,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_DATE,06/24/2014
...,...,...,...,...,...,...,...,...,...
110166,114192149,2026-05-28,Charleston,Kuhaulua,1975-12-21,STD,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 =1> MSH.1=| MSH.2=^~\&amp;# MSH.3.1=Ce...
110177,114189249,2026-06-08,Fredie,Pasion,1967-12-19,General,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 =1> MSH.1=| MSH.2=^~\&amp; MSH.3.1=DLS...
110178,114189249,2026-06-08,Fredie,Pasion,1967-12-19,General,ADMINISTRATIVE,REPORTING_SOURCE_DATE,06/03/2026
110184,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,SIGNAL_DATA,ELR_SIGNAL,ORU_R01 =1> MSH.1=| MSH.2=^~\&amp; MSH.3.1=RRE...


In [17]:
# Read the second file that contains UAT cases
df_uat = pd.read_csv('input/HI Data Validation.csv',na_filter=False)
len(df_uat)

648

In [18]:
df_uat.columns

Index(['Model', 'Source Field Name', 'Source Label', 'CSTE Label',
       'Unnamed: 4', 'Target Table', 'Target Field ID', 'Target Field Label',
       'Case ID', 'Mapped (Y/N)', 'Value Mapped (Y/N/NA)',
       'Verified TEST (Y/N)', 'Verified QA (Y/N)', 'Verified UAT (Y/N)',
       'HI DOH Comments (UAT)', 'InductiveHealth Comments', 'Case ID ALEX',
       'Value', 'Modification Date', 'Questionset', 'Matched Choice ID'],
      dtype='object')

In [19]:
df_uat.loc[:,'Source Field Name'].value_counts(dropna=False)

Source Field Name
DIAGNOSIS_DATE                    9
LABORATORY_TEST_NUMERIC_RESULT    9
STATE                             9
COUNTY                            7
COUNTRY                           6
                                 ..
IS_PERSON_DEAD                    1
STATE_OF_BIRTH                    1
DATE_OF_ENTRY_TO_US               1
GENDER_BIRTH                      1
WORKPLACE_TYPE                    1
Name: count, Length: 458, dtype: int64

In [20]:
# Clean up questions or data elements
df_uat.loc[:,'Source Field Name'] =[ clean_up(q) for q in df_uat.loc[:,'Source Field Name']]

In [21]:
df_uat.loc[:,'Source Field Name'].value_counts(dropna=False)

Source Field Name
DIAGNOSIS_DATE                    9
LABORATORY_TEST_NUMERIC_RESULT    9
STATE                             9
COUNTY                            7
COUNTRY                           6
                                 ..
IS_PERSON_DEAD                    1
STATE_OF_BIRTH                    1
DATE_OF_ENTRY_TO_US               1
GENDER_BIRTH                      1
WORKPLACE_TYPE                    1
Name: count, Length: 458, dtype: int64

In [22]:
df_uat.loc[:,'Model'].value_counts(dropna=False)

Model
General                  220
STD                      219
Hepatitis                 47
Coinfection               43
                          32
Lead                      30
                          19
Childhood Lead            13
STD EHR and screening     12
Aggregate                  7
Cluster                    6
Name: count, dtype: int64

In [23]:
df_uat = df_uat[['Model','Source Field Name', 'Source Label','CSTE Label','Unnamed: 4','Target Table','Target Field ID','Target Field Label']]

In [24]:
# From df_maven only extract the data elements referenced in df_uat

In [25]:
# Merge df_maven with df_uat on QUESTION_ID and Source Field Name
df_merge = pd.merge(df_maven,df_uat, left_on=['QUESTION_ID','model_name'],right_on=['Source Field Name','Model'])

In [26]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34120 entries, 0 to 34119
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   case_id             34120 non-null  object
 1   case_modified_date  34120 non-null  object
 2   first_name          34120 non-null  object
 3   last_name           34120 non-null  object
 4   birth_date          34120 non-null  object
 5   model_name          34120 non-null  object
 6   QUESTIONSET_ID      34120 non-null  object
 7   QUESTION_ID         34120 non-null  object
 8   value               34120 non-null  object
 9   Model               34120 non-null  object
 10  Source Field Name   34120 non-null  object
 11  Source Label        34120 non-null  object
 12  CSTE Label          34120 non-null  object
 13  Unnamed: 4          34120 non-null  object
 14  Target Table        34120 non-null  object
 15  Target Field ID     34120 non-null  object
 16  Target Field Label  34

In [27]:
df_merge = df_merge[['case_id','case_modified_date','first_name','last_name','birth_date','model_name','QUESTIONSET_ID','Source Field Name','Source Label','value','CSTE Label','Unnamed: 4','Target Table','Target Field ID','Target Field Label']]

In [28]:
len(df_merge)

34120

In [29]:
df_merge

,case_id,case_modified_date,first_name,last_name,birth_date,model_name,QUESTIONSET_ID,Source Field Name,Source Label,value,CSTE Label,Unnamed: 4,Target Table,Target Field ID,Target Field Label
0,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_TYPE,,PHC247,Reporting Source Type Code,184,Incidents,CustomText13,Reporting Source Type Code
1,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,BIRTH_DATE,Birth date,12/21/1959,Birth Date,198,Incidents,BirthDate,Birth Date
2,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,JURISDICTION,,HI,National Reporting Jurisdiction,192,Incidents,CustomCode10ID,National Reporting Jurisdiction
3,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,STATE,State,HI,Subject Address State,206,Incidents,ClaimantStateID,Subject Address State
4,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,ZIP,,96793,Subject Address ZIP Code,207,Incidents,ClaimantPostalCode,Subject Address ZIP Code
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34115,114189249,2026-06-08,Fredie,Pasion,1967-12-19,General,ADMINISTRATIVE,MMWR_YEAR,> MMWR Year,2026,MMWR Year,10,Incidents,CustomNumber2,[INV166]MMWR Year
34116,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE,,21,Age at Case Investigation,199,Incidents,CustomNumber7,Age at Case Investigation
34117,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE_UNITS,,YEARS,Age Unit at Case Investigation,200,Incidents,CustomCode33ID,Age Unit at Case Investigation
34118,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,ADMINISTRATIVE,MMWR_WEEK,,21,MMWR Week,180,Incidents,CustomText18,MMWR Week


In [30]:
df_merge.rename(columns={'Unnamed: 4':'question_number','value':'Value'},inplace=True)

In [31]:
df_merge

,case_id,case_modified_date,first_name,last_name,birth_date,model_name,QUESTIONSET_ID,Source Field Name,Source Label,Value,CSTE Label,question_number,Target Table,Target Field ID,Target Field Label
0,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_TYPE,,PHC247,Reporting Source Type Code,184,Incidents,CustomText13,Reporting Source Type Code
1,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,BIRTH_DATE,Birth date,12/21/1959,Birth Date,198,Incidents,BirthDate,Birth Date
2,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,JURISDICTION,,HI,National Reporting Jurisdiction,192,Incidents,CustomCode10ID,National Reporting Jurisdiction
3,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,STATE,State,HI,Subject Address State,206,Incidents,ClaimantStateID,Subject Address State
4,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,ZIP,,96793,Subject Address ZIP Code,207,Incidents,ClaimantPostalCode,Subject Address ZIP Code
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34115,114189249,2026-06-08,Fredie,Pasion,1967-12-19,General,ADMINISTRATIVE,MMWR_YEAR,> MMWR Year,2026,MMWR Year,10,Incidents,CustomNumber2,[INV166]MMWR Year
34116,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE,,21,Age at Case Investigation,199,Incidents,CustomNumber7,Age at Case Investigation
34117,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE_UNITS,,YEARS,Age Unit at Case Investigation,200,Incidents,CustomCode33ID,Age Unit at Case Investigation
34118,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,ADMINISTRATIVE,MMWR_WEEK,,21,MMWR Week,180,Incidents,CustomText18,MMWR Week


In [32]:
df_merge['Mapped (Y/N)']=""
df_merge['Value Mapped (Y/N/NA)']=""
df_merge['Verified TEST (Y/N)']=""
df_merge['Verified QA (Y/N)']=""
df_merge['Verified UAT (Y/N)'] = ""
df_merge['HI DOH Comments (UAT)']=""
df_merge['InductiveHealth Comments']=""

In [33]:
df_merge

,case_id,case_modified_date,first_name,last_name,birth_date,model_name,QUESTIONSET_ID,Source Field Name,Source Label,Value,...,Target Table,Target Field ID,Target Field Label,Mapped (Y/N),Value Mapped (Y/N/NA),Verified TEST (Y/N),Verified QA (Y/N),Verified UAT (Y/N),HI DOH Comments (UAT),InductiveHealth Comments
0,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,REPORTING_SOURCE_TYPE,,PHC247,...,Incidents,CustomText13,Reporting Source Type Code,,,,,,,
1,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,BIRTH_DATE,Birth date,12/21/1959,...,Incidents,BirthDate,Birth Date,,,,,,,
2,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,ADMINISTRATIVE,JURISDICTION,,HI,...,Incidents,CustomCode10ID,National Reporting Jurisdiction,,,,,,,
3,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,STATE,State,HI,...,Incidents,ClaimantStateID,Subject Address State,,,,,,,
4,100362493,2026-04-16,Champ,Wright,1959-12-21,Hepatitis,DEMOGRAPHIC,ZIP,,96793,...,Incidents,ClaimantPostalCode,Subject Address ZIP Code,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34115,114189249,2026-06-08,Fredie,Pasion,1967-12-19,General,ADMINISTRATIVE,MMWR_YEAR,> MMWR Year,2026,...,Incidents,CustomNumber2,[INV166]MMWR Year,,,,,,,
34116,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE,,21,...,Incidents,CustomNumber7,Age at Case Investigation,,,,,,,
34117,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,DEMOGRAPHIC,AGE_UNITS,,YEARS,...,Incidents,CustomCode33ID,Age Unit at Case Investigation,,,,,,,
34118,114191082,2026-06-13,Joie,Miyashiro,2005-01-31,Hepatitis,ADMINISTRATIVE,MMWR_WEEK,,21,...,Incidents,CustomText18,MMWR Week,,,,,,,


In [34]:
df_merge.to_csv('output/case_samples_uat_testing_20260721.csv',index=False)